In [1]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import numpy as np
import editdistance
import torch.nn.functional as F
import random

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
base_dir = os.getcwd()

ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_transliteration_train.txt') 

images_dir = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_word_train')

In [4]:
filenames = []
labels = []

# Read the ground truth text file
with open(ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # Ensure the line is not empty
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                filenames.append(filename)
                labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the data
data = pd.DataFrame({
    'filename': filenames,
    'label': labels
})

# Display the DataFrame
data.head()


,filename,label
0,train1.png,kagastu
1,train2.png,","
2,train3.png,gelak
3,train4.png,ancak
4,train5.png,","


In [5]:
all_text = ''.join(data['label'])
chars = sorted(list(set(all_text)))  # unique characters

char_to_idx = {}
char_to_idx['<BLANK>'] = 0  # blank for CTC
char_to_idx['<PAD>'] = 1    # optional pad index

start_index = 2
for i, char in enumerate(chars, start=start_index):
    char_to_idx[char] = i

idx_to_char = {idx: char for char, idx in char_to_idx.items()}

vocab_size = len(char_to_idx)
print(f"Vocabulary size: {vocab_size}")


Vocabulary size: 56


In [6]:
# print("Character to Index Mapping:")
# for char, idx in char_to_idx.items():
#     print(f"'{char}': {idx}")


In [7]:
# empty_labels = data[data['label'].str.strip() == '']
# print(f"Number of empty labels in training data: {len(empty_labels)}")

In [8]:
def encode_label_ctc(label, char_to_idx, max_length):
    encoded = [char_to_idx.get(ch, 1)  # if unknown char, use <PAD> or define <UNK> if you prefer
               for ch in label]
    if len(encoded) < max_length:
        encoded += [char_to_idx['<PAD>']] * (max_length - len(encoded))
    else:
        encoded = encoded[:max_length]
    return encoded

# pick a max_label_length with some margin. +2 is optional if you want extra pad.
max_label_length = max(len(lbl) for lbl in data['label']) + 2
print(f"Max label length: {max_label_length}")

data['encoded_label'] = data['label'].apply(
    lambda x: encode_label_ctc(x, char_to_idx, max_label_length)
)

Max label length: 18


In [9]:
# Define a custom dataset class
# Image data
class BalineseDataset(Dataset):
    def __init__(self, data, images_dir, transform=None):
        self.data = data.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """given index, return corresponding image and label"""
        img_name = self.data.loc[idx, 'filename']
        label = self.data.loc[idx, 'encoded_label']
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label


In [10]:
# Define image transformations
transform = transforms.Compose([
    transforms.Resize((64, 256)),  # Resize images to a fixed size
    transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5),  # Mean for R, G, B channels
                            (0.5, 0.5, 0.5))  # Std for R, G, B channels  # Normalize images
])

In [11]:
train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)

train_dataset = BalineseDataset(train_data, images_dir, transform=transform)
val_dataset   = BalineseDataset(val_data, images_dir, transform=transform)

# Create data loaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# train_loader = train_loader.to(devie)
# val_loader = val_loader.to(device)


In [12]:
class CRNN(nn.Module):
    def __init__(self, num_classes, hidden_dim=256, num_lstm_layers=2):
        super(CRNN, self).__init__()
        resnet = models.resnet18(pretrained=True)
        layers = list(resnet.children())[:-2]  # remove avgpool & fc
        self.cnn = nn.Sequential(*layers)

        # Pool to height=1
        self.pool = nn.AdaptiveAvgPool2d((1, None))

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=512,  # from resnet output channels
            hidden_size=hidden_dim,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True
        )

        # Final linear
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: (batch, 3, 64, 256)
        features = self.cnn(x)                  # -> (batch, 512, h', w')
        features = self.pool(features)          # -> (batch, 512, 1, w'')
        features = features.squeeze(2)          # -> (batch, 512, w'')
        features = features.permute(0, 2, 1)    # -> (batch, w'', 512)

        lstm_out, _ = self.lstm(features)       # -> (batch, w'', hidden_dim*2)
        logits = self.fc(lstm_out)              # -> (batch, w'', num_classes)

        # for CTC: (seq_len, batch, num_classes)
        logits = logits.permute(1, 0, 2)
        return logits



In [13]:
num_classes = vocab_size  # including blank
model = CRNN(num_classes=num_classes)
model = model.to(device)

ctc_loss_fn = nn.CTCLoss(blank=0, reduction='mean')  # 0 as blank index
optimizer = optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 74.8MB/s]


In [14]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images = images.cuda()
        labels = labels.cuda()

        logits = model(images) # (seq_len, batch, num_classes)
        seq_len, batch_size, _ = logits.size()

        # input_lengths: each sample has seq_len timesteps
        input_lengths = torch.full(
            size=(batch_size,),
            fill_value=seq_len,
            dtype=torch.long
        ).cuda()

        # Flatten labels & compute target_lengths
        target_list = []
        lengths_list = []
        for i in range(batch_size):
            raw = labels[i].cpu().tolist()
            cleaned = [r for r in raw if r != char_to_idx['<PAD>']]  # remove <PAD>
            target_list.extend(cleaned)
            lengths_list.append(len(cleaned))

        targets = torch.tensor(target_list, dtype=torch.long).cuda()
        target_lengths = torch.tensor(lengths_list, dtype=torch.long).cuda()

        loss = ctc_loss_fn(logits, targets, input_lengths, target_lengths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def ctc_greedy_decode(logits, blank=0):
    # logits: (seq_len, batch, num_classes)
    seq_len, batch_size, num_classes = logits.size()
    softmaxed = F.log_softmax(logits, dim=2)  # or F.softmax
    max_probs = softmaxed.argmax(dim=2)       # (seq_len, batch)

    all_preds = []
    for b in range(batch_size):
        raw_seq = max_probs[:, b].tolist()    # length = seq_len
        deduped = []
        prev = None
        for char_idx in raw_seq:
            # skip blank and duplicates
            if char_idx != blank and char_idx != prev:
                deduped.append(char_idx)
            prev = char_idx
        all_preds.append(deduped)
    return all_preds

def validate(model, loader):
    model.eval()
    total_loss = 0.0
    total_cer = 0.0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.cuda()
            labels = labels.cuda()

            logits = model(images)
            seq_len, batch_size, _ = logits.size()

            input_lengths = torch.full(
                size=(batch_size,),
                fill_value=seq_len,
                dtype=torch.long
            ).cuda()

            target_list = []
            lengths_list = []
            for i in range(batch_size):
                raw = labels[i].cpu().tolist()
                cleaned = [r for r in raw if r != char_to_idx['<PAD>']]
                target_list.extend(cleaned)
                lengths_list.append(len(cleaned))

            targets = torch.tensor(target_list, dtype=torch.long).cuda()
            target_lengths = torch.tensor(lengths_list, dtype=torch.long).cuda()

            loss = ctc_loss_fn(logits, targets, input_lengths, target_lengths)
            total_loss += loss.item()

            # decode and compute CER
            pred_indices = ctc_greedy_decode(logits, blank=char_to_idx['<BLANK>'])
            for b in range(batch_size):
                # predicted
                pred_str = ''.join(idx_to_char[idx] for idx in pred_indices[b])

                # ground truth
                raw_gt = [r for r in labels[b].cpu().tolist() if r != char_to_idx['<PAD>']]
                gt_str = ''.join(idx_to_char[r] for r in raw_gt)

                if len(gt_str) > 0:
                    cer = editdistance.eval(pred_str, gt_str) / len(gt_str)
                    total_cer += cer
                total_samples += 1

    avg_loss = total_loss / len(loader)
    avg_cer = total_cer / total_samples if total_samples > 0 else 0.0
    return avg_loss, avg_cer

In [15]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_cer = validate(model, val_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"- Train Loss: {train_loss:.4f}, "
          f"- Val Loss: {val_loss:.4f}, "
          f"- Val CER: {val_cer:.4f}")

Epoch [1/5] - Train Loss: nan, - Val Loss: nan, - Val CER: 1.0000
Epoch [2/5] - Train Loss: nan, - Val Loss: nan, - Val CER: 1.0000
Epoch [3/5] - Train Loss: nan, - Val Loss: nan, - Val CER: 1.0000
Epoch [4/5] - Train Loss: nan, - Val Loss: nan, - Val CER: 1.0000
Epoch [5/5] - Train Loss: nan, - Val Loss: nan, - Val CER: 1.0000


In [16]:
# Paths to test data
test_ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_transliteration_test.txt')
test_images_dir = os.path.join(base_dir, '/kaggle/input/balinese-dataset/data/balinese_word_test')

test_filenames = []
test_labels = []

# Read the ground truth text file for the test set
with open(test_ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                test_filenames.append(filename)
                test_labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the test data
test_data = pd.DataFrame({
    'filename': test_filenames,
    'label': test_labels
})


In [17]:
max_label_length_test = max(len(lbl) for lbl in test_data['label']) + 2
test_data['encoded_label'] = test_data['label'].apply(
    lambda x: encode_label_ctc(x, char_to_idx, max_label_length_test)
)

In [18]:
# Define the same transformations used during training
test_transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Create test dataset
test_dataset = BalineseDataset(test_data, test_images_dir, transform=test_transform)

# Create test data loader
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


In [19]:
model.eval()
test_results = []
total_edit_dist = 0
total_ref_len  = 0

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.cuda()
        labels = labels.cuda()

        logits = model(images) # (seq_len, batch, num_classes)
        seq_len, batch_size, _ = logits.size()

        pred_indices = ctc_greedy_decode(logits, blank=char_to_idx['<BLANK>'])

        for i in range(batch_size):
            pred_str = ''.join(idx_to_char[idx] for idx in pred_indices[i])

            raw_gt = [r for r in labels[i].cpu().tolist() if r != char_to_idx['<PAD>']]
            gt_str = ''.join(idx_to_char[r] for r in raw_gt)

            # compute CER
            if len(gt_str) > 0:
                dist = editdistance.eval(pred_str, gt_str)
                total_edit_dist += dist
                total_ref_len += len(gt_str)
            else:
                # no ground truth
                pass

            # Store for debugging
            image_filename = test_data.iloc[batch_idx * batch_size + i]['filename']
            test_results.append({
                'image_filename': image_filename,
                'predicted_caption': pred_str,
                'ground_truth_caption': gt_str
            })

if total_ref_len > 0:
    total_cer = total_edit_dist / total_ref_len
else:
    total_cer = 0.0

print(f"Test CER: {total_cer:.4f}")

Test CER: 1.0000


In [22]:
# Additionally, print some random samples to get a general idea of model performance
import random

M = 5  # Number of random samples to display
random_samples = random.sample(test_results, M)
print(f"\nRandom {M} samples from Test Set:")
for i, result in enumerate(random_samples):
    print(f"Sample {i + 1}:")
    print(f"Image Filename: {result['image_filename']}")
    print(f"CER: {result['cer']:.4f}")
    print(f"Ground Truth Caption: {result['ground_truth_caption']}")
    print(f"Predicted Caption  : {result['predicted_caption']}\n")


Random 5 samples from Test Set:
